In [38]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Visual Analytics & Virality Prediction

Every single second, thousands of images are uploaded to attention-driven platforms like Instagram and TikTok. While traditional digital marketing models heavily focus on text metadata—such as hashtags, captions, and posting time—modern recommendation algorithms prioritize a visual-first approach.This notebook explores a fundamental question: Can we predict if an image will trend based purely on its raw mathematical pixel properties?Instead of using heavy, black-box deep learning architectures, this project focuses on clear statistical inference. We treat every image as a raw 3D matrix tensor ($I \in \mathbb{R}^{M \times N \times 3}$) and extract three key visual indicators:

1. Color Saturation: The structural richness and vibrancy of the color palette.

2. Visual Clutter (Edge Density): The complexity and minimalist vs. chaotic nature of the composition.

3. Brightness Contrast: The variance of lighting and depth across the image matrix.

Using a part of the open-source AVA (Aesthetic Visual Analysis) dataset, I will mathematically map out the differences between high-engagement (viral) images and low-performing ones, ultimately building an interpretable statistical logistic regression model to calculate a post's probability of trending.

## Mathematical Formulation

### The Input Space (Image Tensors)

A digital image is not merely a graphic file; it is a multi-dimensional real-valued tensor:
$$I \in \mathbb{R}^{M \times N \times 3}$$

Where:
- $M$ represents the discrete row index (pixel height).
- $N$ represents the discrete column index (pixel width).
- $3$ represents the color channel depth mapped to the normalized RGB/HSV vector space.

### The Feature Mapping Function

Defining a vector-valued feature extraction function $f(I)$ that compresses the high-dimensional image tensor into a low-dimensional space containing the three target visual metrics:
$$f(I) = \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} \mu_{\text{(saturation)}} \\ \rho_{\text{(clutter)}} \\ \sigma_{\text{(contrast)}} \end{bmatrix}$$

This function maps our data from $\mathbb{R}^{M \times N \times 3} \rightarrow \mathbb{R}^3$, allowing feeding traditional statistical models without encountering a problem with the dimensionality (running out of computing power because there are too many data points).

### The Target Space (Engagement Optimization)

Defining the target domain as a discrete binary set:
$$Y \in \{0, 1\}$$
Where $Y = 1$ indicates a "viral" image (top-tier community aesthetic rating) and $Y = 0$ indicates a non-trending one. The ultimate objective is to approximate the true conditional probability distribution:
$$P(Y = 1 \mid X)$$

## Data Pipeline & Local Batch Processing

In [42]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATASET_DIR = r"C:\Users\ivona\Downloads\VP" 
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
METADATA_PATH = os.path.join(DATASET_DIR, "ground_truth_dataset.csv")

In [61]:
columns = [
    'image_id', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 
    'tag1', 'tag2', 'challenge_id'
]
df_raw = pd.read_csv(METADATA_PATH, sep=',')

In [64]:
# Calculate true continuous aesthetic means via vector dot-product
vote_columns = [f'vote_{i}' for i in range(1, 11)]
votes = df_raw[vote_columns].values
score_weights = np.arange(1, 11)
df_raw['mean_score'] = np.sum(votes * score_weights, axis=1) / np.sum(votes, axis=1)

In [65]:
# Establish binary target classification matrix threshold (>= 5.5 = Viral)
df_raw['is_viral'] = (df_raw['mean_score'] >= 5.5).astype(int)

In [68]:
# Map dynamic absolute paths for physical disk checks
df_raw['image_id_clean'] = df_raw['image_num'].astype(str)
df_raw['file_path'] = df_raw['image_id_clean'].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))

print(f"\n-> Raw catalog parsed: {len(df_raw):,} entries loaded.")


-> Raw catalog parsed: 255,508 entries loaded.


In [69]:
print("\nValidating physical image file records on disk...")
df_existing = df_raw[df_raw['file_path'].apply(os.path.exists)].copy()
print(f"-> Verified available assets: {len(df_existing):,} files present.")


Validating physical image file records on disk...
-> Verified available assets: 255,508 files present.
